# Notebook 03 — Modèle Hiérarchique Bayésien (Approche A)

> **Quand utiliser ce notebook ?**  
> C'est l'approche la plus riche mais aussi la plus complexe. À utiliser **après** avoir validé Transfer Learning (NB 04) et Mixed Effects (NB 02), dans les cas où :
> - Les parties prenantes ont besoin d'**intervalles de confiance formels** sur chaque prédiction
> - Certaines familles ont > 60% de variantes rares et le Mixed Effects converge mal
> - Tu veux intégrer de la **connaissance experte** de façon formelle et traçable (prior informé)
>
> **Prérequis** : NB 04 (Transfer Learning validé) + NB 02 (Mixed Effects compris)

---

## Concepts abordés
1. Théorème de Bayes appliqué au costing
2. Estimateur de James-Stein : le shrinkage en une formule
3. Implémentation du modèle hiérarchique avec NumPy (version pédagogique)
4. Version complète avec PyMC
5. Visualisation des posteriors et intervalles de crédibilité

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_parquet(Path('data/dataset_industriel.parquet'))
print(f"Dataset : {len(df):,} lignes")

## 1. Le théorème de Bayes pour le costing

**Question** : quelle est la vraie moyenne de coût $\mu_v$ de la variante $v$ ?

Bayes répond :

$$\underbrace{P(\mu_v \mid \text{données})}_{\text{posterior}} \propto \underbrace{P(\text{données} \mid \mu_v)}_{\text{vraisemblance}} \times \underbrace{P(\mu_v)}_{\text{prior}}$$

**Le prior** encode ce qu'on sait *avant* de voir les données de la variante.  
Pour nous : le coût de la variante est probablement proche du coût moyen de sa famille.

$$\mu_v \sim \mathcal{N}(\mu_f, \sigma_f^2) \quad \text{avec } \mu_f = \text{coût moyen de la famille } f$$

**La vraisemblance** encode ce qu'on observe :

$$y_{vt} \sim \mathcal{N}(\mu_v, \sigma_{\varepsilon}^2)$$

**Le posterior** (analytique dans le cas Gaussien) est :

$$\hat{\mu}_v = \underbrace{\lambda}_{\text{poids données}} \cdot \bar{y}_v + \underbrace{(1-\lambda)}_{\text{poids prior}} \cdot \mu_f$$

$$\lambda = \frac{n_v}{n_v + \sigma_\varepsilon^2/\sigma_f^2}$$

## 2. L'estimateur de James-Stein : le shrinkage en une formule

In [ ]:
def bayesian_shrinkage(y_obs: np.ndarray, mu_prior: float,
                        sigma_prior: float, sigma_eps: float) -> tuple:
    """
    Calcule l'estimateur posterior bayésien Gaussien.

    Returns: (mu_post, sigma_post, lambda_)
    """
    n        = len(y_obs)
    y_bar    = y_obs.mean()
    lambda_  = n / (n + (sigma_eps / sigma_prior) ** 2)
    mu_post  = lambda_ * y_bar + (1 - lambda_) * mu_prior
    sigma_post = np.sqrt(1 / (n / sigma_eps**2 + 1 / sigma_prior**2))
    return mu_post, sigma_post, lambda_


MU_FAMILLE   = 1000.0
SIGMA_FAMILLE = 120.0
SIGMA_EPS     =  80.0

n_values = [1, 2, 3, 5, 10, 20, 50]
results  = []

for n in n_values:
    y_sim = np.random.normal(1100, SIGMA_EPS, n)  # vraie moyenne 1100€
    mu_post, sigma_post, lam = bayesian_shrinkage(
        y_sim, MU_FAMILLE, SIGMA_FAMILLE, SIGMA_EPS
    )
    results.append({'n_obs': n, 'y_bar': y_sim.mean(), 'mu_post': mu_post,
                    'sigma_post': sigma_post, 'lambda': lam})

df_shrink = pd.DataFrame(results)
print("Évolution du shrinkage avec le nombre d'observations :")
print(df_shrink.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(df_shrink['n_obs'], df_shrink['lambda'], 'o-', color='#3498db', lw=2.5, ms=8)
ax.axhline(0, color='#e74c3c', linestyle='--', label='Full prior (λ=0)')
ax.axhline(1, color='#27ae60', linestyle='--', label='Full data (λ=1)')
ax.set_xlabel("Nombre d'observations")
ax.set_ylabel('λ (poids des données)')
ax.set_title('Shrinkage : poids accordé aux données vs au prior famille')
ax.legend()
ax.set_ylim(0, 1.1)

ax = axes[1]
ax.plot(df_shrink['n_obs'], df_shrink['y_bar'], 's--', color='#e74c3c',
        label='Moyenne brute (y̅)', lw=2, ms=8)
ax.plot(df_shrink['n_obs'], df_shrink['mu_post'], 'o-', color='#3498db',
        label='Estimateur bayésien (μ_post)', lw=2, ms=8)
ax.fill_between(df_shrink['n_obs'],
                df_shrink['mu_post'] - 1.96 * df_shrink['sigma_post'],
                df_shrink['mu_post'] + 1.96 * df_shrink['sigma_post'],
                alpha=0.2, color='#3498db', label='IC 95%')
ax.axhline(1100, color='black', linestyle=':', lw=2, label='Vraie valeur (1100€)')
ax.axhline(MU_FAMILLE, color='gray', linestyle=':', lw=1,
           label=f'Prior famille ({MU_FAMILLE:.0f}€)')
ax.set_xlabel("Nombre d'observations")
ax.set_ylabel('Coût estimé (€)')
ax.set_title('Estimation bayésienne vs moyenne brute')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/fig_bayesian_shrinkage.png', dpi=150)
plt.show()

**Lecture** : avec 1-3 obs, l'estimateur bayésien est plus proche de la vraie valeur (1100€) que la moyenne brute. À partir de ~20 obs, les deux convergent.

## 3. Application au dataset complet (Empirical Bayes)

In [ ]:
# Estimation des hyperparamètres par famille
variante_means = df.groupby(['variante', 'famille', 'n_obs_variante'])['cout'].mean().reset_index()
variante_means.rename(columns={'cout': 'cout_moyen'}, inplace=True)

famille_params = variante_means.groupby('famille')['cout_moyen'].agg(
    mu_famille='mean', sigma_famille='std'
).fillna(100)

matures = df[df['n_obs_variante'] >= 25]
sigma_eps_global = matures.groupby('variante')['cout'].std().mean()

print(f"Bruit résiduel estimé (σ_ε) : {sigma_eps_global:.1f}€")

# Application à toutes les variantes
bayes_estimates = []
for _, row in variante_means.iterrows():
    fp = famille_params.loc[row['famille']]
    obs = df[df['variante'] == row['variante']]['cout'].values
    mu_post, sigma_post, lam = bayesian_shrinkage(
        obs, fp['mu_famille'], max(fp['sigma_famille'], 10), sigma_eps_global
    )
    bayes_estimates.append({
        'variante': row['variante'], 'famille': row['famille'],
        'n_obs': row['n_obs_variante'], 'cout_brut': row['cout_moyen'],
        'cout_bayes': mu_post, 'incertitude': sigma_post,
        'lambda': lam, 'mu_famille': fp['mu_famille'],
    })

df_bayes = pd.DataFrame(bayes_estimates)
df_bayes['categorie'] = pd.cut(
    df_bayes['n_obs'], bins=[0, 5, 24, 200],
    labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)']
)

print("\nλ moyen par catégorie (plus proche de 0 = plus de shrinkage) :")
print(df_bayes.groupby('categorie')['lambda'].describe().round(3))

In [ ]:
# Visualisation sur une famille exemple
famille_ex = df_bayes['famille'].value_counts().index[0]
df_fam = df_bayes[df_bayes['famille'] == famille_ex].sort_values('n_obs')

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_fam))
mu_fam = df_fam['mu_famille'].iloc[0]

ax.scatter(x, df_fam['cout_brut'], color='#e74c3c', s=80, zorder=5,
           label='Moyenne brute (y̅ variante)')
ax.scatter(x, df_fam['cout_bayes'], color='#3498db', s=80, zorder=5,
           label='Estimateur bayésien (μ_post)')
ax.errorbar(x, df_fam['cout_bayes'],
            yerr=1.96 * df_fam['incertitude'],
            fmt='none', color='#3498db', alpha=0.4, capsize=4)
ax.axhline(mu_fam, color='gray', linestyle='--', lw=1.5,
           label=f'Moyenne famille ({mu_fam:.0f}€)')

ax.set_xticks(x)
ax.set_xticklabels([f"n={r['n_obs']}" for _, r in df_fam.iterrows()],
                    rotation=45, ha='right')
ax.set_xlabel('Variante (triée par n_obs)')
ax.set_ylabel('Coût moyen estimé (€)')
ax.set_title(f'Famille {famille_ex} — Shrinkage bayésien\n'
             '(les rares convergent vers la moyenne famille)')
ax.legend()

plt.tight_layout()
plt.savefig('data/fig_bayesian_family.png', dpi=150)
plt.show()

## 4. (Optionnel) Version PyMC avec MCMC

In [ ]:
# --- CODE PyMC (décommenter après : pip install pymc arviz) ---
#
# import pymc as pm
# import arviz as az
#
# variante_idx, variantes = pd.factorize(df['variante'])
# variante_to_famille = df.groupby('variante')['famille'].first()
# famille_idx, familles = pd.factorize(variante_to_famille)
#
# with pm.Model() as hierarchical_model:
#     mu_global    = pm.Normal('mu_global', mu=1000, sigma=500)
#     sigma_global = pm.HalfNormal('sigma_global', sigma=200)
#
#     mu_famille   = pm.Normal('mu_famille', mu=mu_global,
#                               sigma=sigma_global, shape=len(familles))
#     sigma_famille = pm.HalfNormal('sigma_famille', sigma=100, shape=len(familles))
#
#     mu_variante  = pm.Normal('mu_variante',
#                               mu=mu_famille[famille_idx],
#                               sigma=sigma_famille[famille_idx],
#                               shape=len(variantes))
#
#     sigma_obs = pm.HalfNormal('sigma_obs', sigma=100)
#     y_obs = pm.Normal('y_obs',
#                        mu=mu_variante[variante_idx],
#                        sigma=sigma_obs,
#                        observed=df['cout'].values)
#
#     trace = pm.sample(1000, tune=1000, target_accept=0.9, return_inferencedata=True)
#
# az.plot_posterior(trace, var_names=['mu_global', 'sigma_global'])

print("Pour PyMC complet : pip install pymc arviz")
print("La version analytique (Section 3) est équivalente pour des distributions Gaussiennes.")

## Résumé du Notebook 03

| Concept | Formule clé |
|---------|-------------|
| Prior famille | $\mu_v \sim \mathcal{N}(\mu_f, \sigma_f^2)$ |
| Posterior | $\hat{\mu}_v = \lambda \cdot \bar{y}_v + (1-\lambda) \cdot \mu_f$ |
| Shrinkage | $\lambda = n_v / (n_v + \sigma_\varepsilon^2/\sigma_f^2)$ |
| n petit | $\lambda \to 0$ → on fait confiance au prior famille |
| n grand | $\lambda \to 1$ → on fait confiance aux données locales |
| IC bayésien | Se réduit automatiquement avec plus d'observations |

**→ Notebook suivant : [04_transfer_learning.ipynb](04_transfer_learning.ipynb)**